# PET-KLS 3D edge network

Interactive Schaefer-100 / Yeo-17 connectome from edge-wise Mann–Whitney U results.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from nilearn import datasets, plotting
from nilearn.image import load_img
from nilearn.plotting import find_parcellation_cut_coords

# Paths are relative to the notebook's working directory.
EDGES_CSV = Path("data/edges_mwu_bh_all.csv")
OUTPUT_HTML = Path("docs/index.html")
OUTPUT_EDGES_CSV = Path("outputs/selected_edges.csv")

Q_THRESHOLD = 0.05
VIEW_MODE = "top"             # "top" or "all"
TOP_POSITIVE = 30             # used when VIEW_MODE == "top"
TOP_NEGATIVE = 30
MIN_ABS_EFFECT = 0.0
EDGE_VALUE_GAIN = 1.0        # visual contrast only; 1.0 preserves effect values
NODE_SIZE = 10

EFFECT_COL = "rank_biserial_r"
P_COL = "p"
Q_COL = "q"

NETWORK_COLORS = {
    "VisCent": "#781286",
    "VisPeri": "#FF0000",
    "SomMotA": "#4682B4",
    "SomMotB": "#2ACCA4",
    "DorsAttnA": "#4A9B3C",
    "DorsAttnB": "#00760E",
    "SalVentAttnA": "#C43AFA",
    "SalVentAttnB": "#FF98D5",
    "LimbicA": "#DCF8A4",
    "LimbicB": "#7A8732",
    "ContA": "#778CB0",
    "ContB": "#E69422",
    "ContC": "#87324A",
    "TempPar": "#0C30FF",
    "DefaultA": "#000082",
    "DefaultB": "#FFFF00",
    "DefaultC": "#CD3E4E",
}


In [ ]:
def canonical_roi(label):
    name = str(label).strip()
    if name.startswith(("b'", 'b"')):
        name = name[2:-1]
    name = name.replace("17Networks_", "").replace("7Networks_", "")
    return name.replace("-", "_").lower()


def network_from_label(label):
    core = str(label).replace("17Networks_", "").replace("7Networks_", "")
    parts = core.split("_")
    if len(parts) < 2 or parts[1] not in NETWORK_COLORS:
        raise ValueError(f"Unrecognized Schaefer-17 network label: {label}")
    return parts[1]


def read_edges(path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing edge results: {path}")
    data = pd.read_csv(path)
    data.columns = data.columns.str.strip()
    required = {"label_i", "label_j", EFFECT_COL, P_COL, Q_COL}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing edge columns: {sorted(missing)}")
    for col in (EFFECT_COL, P_COL, Q_COL):
        data[col] = pd.to_numeric(data[col], errors="coerce")
    if data[list(required)].isna().any().any():
        raise ValueError("Edge results contain missing labels, effects, p-values or q-values.")
    if not data[P_COL].between(0, 1).all() or not data[Q_COL].between(0, 1).all():
        raise ValueError("p-values and q-values must be in [0, 1].")
    if not data[EFFECT_COL].between(-1, 1).all():
        raise ValueError("Rank-biserial effect sizes must be in [-1, 1].")
    if (data["label_i"].astype(str) == data["label_j"].astype(str)).any():
        raise ValueError("Self-edges are not valid undirected ROI pairs.")
    data["roi_i"] = data["label_i"].map(canonical_roi)
    data["roi_j"] = data["label_j"].map(canonical_roi)
    data["edge_key"] = data.apply(
        lambda row: tuple(sorted((row["roi_i"], row["roi_j"]))), axis=1
    )
    data = data.sort_values([Q_COL, P_COL], kind="mergesort")
    # Preserve exactly one statistic per unique undirected ROI pair.
    duplicated = data.duplicated("edge_key", keep=False)
    if duplicated.any():
        repeated = data.loc[duplicated].groupby("edge_key")
        for _, group in repeated:
            if (group[EFFECT_COL].max() - group[EFFECT_COL].min() > 1e-8) or (
                group[Q_COL].max() - group[Q_COL].min() > 1e-8
            ):
                raise ValueError("Conflicting statistics for a symmetric ROI pair.")
        data = data.drop_duplicates("edge_key", keep="first")
    return data.reset_index(drop=True)


def choose_edges(data):
    sig = data[
        (data[Q_COL] < Q_THRESHOLD)
        & (data[EFFECT_COL].abs() >= MIN_ABS_EFFECT)
        & (data[EFFECT_COL] != 0)
    ].copy()
    if sig.empty:
        raise ValueError("No edges satisfy the configured significance/effect thresholds.")
    if VIEW_MODE == "top":
        positive = sig[sig[EFFECT_COL] > 0].head(TOP_POSITIVE)
        negative = sig[sig[EFFECT_COL] < 0].head(TOP_NEGATIVE)
        sig = pd.concat([positive, negative], ignore_index=True)
    elif VIEW_MODE != "all":
        raise ValueError("VIEW_MODE must be 'top' or 'all'.")
    if sig.empty:
        raise ValueError("No edges selected for rendering.")
    return sig


def schaefer100_coordinates():
    atlas = datasets.fetch_atlas_schaefer_2018(
        n_rois=100, yeo_networks=17, resolution_mm=2
    )
    labels = [
        value.decode("utf-8") if isinstance(value, bytes) else str(value)
        for value in atlas.labels
    ]
    if len(labels) == 101 and labels[0].lower() == "background":
        labels = labels[1:]
    if len(labels) != 100:
        raise ValueError(f"Unexpected Schaefer atlas label count: {len(labels)}")
    coords = find_parcellation_cut_coords(labels_img=load_img(atlas.maps))
    if coords.shape != (100, 3):
        raise ValueError(f"Unexpected Schaefer atlas coordinates: {coords.shape}")
    canonical = [canonical_roi(label) for label in labels]
    if len(set(canonical)) != 100:
        raise ValueError("Schaefer atlas contains duplicate ROI labels.")
    return {
        key: (original, coordinates)
        for key, original, coordinates in zip(canonical, labels, coords, strict=True)
    }


def build_connectome(edges, atlas_lookup):
    used = set(edges["roi_i"]).union(set(edges["roi_j"]))
    missing = used.difference(atlas_lookup)
    if missing:
        raise ValueError(f"Edge labels not found in Schaefer-100 atlas: {sorted(missing)[:12]}")
    nodes = sorted(used)
    index = {node: i for i, node in enumerate(nodes)}
    coordinates = np.vstack([atlas_lookup[node][1] for node in nodes])
    node_colors = [NETWORK_COLORS[network_from_label(atlas_lookup[node][0])] for node in nodes]
    adjacency = np.zeros((len(nodes), len(nodes)), dtype=float)
    for row in edges.itertuples(index=False):
        i, j = index[row.roi_i], index[row.roi_j]
        value = float(np.clip(getattr(row, EFFECT_COL) * EDGE_VALUE_GAIN, -1.0, 1.0))
        adjacency[i, j] = value
        adjacency[j, i] = value
    if not np.allclose(adjacency, adjacency.T) or np.any(np.diag(adjacency) != 0):
        raise RuntimeError("Connectome matrix validation failed.")
    return adjacency, coordinates, node_colors


In [ ]:
edge_results = read_edges(EDGES_CSV)
selected = choose_edges(edge_results)
atlas_lookup = schaefer100_coordinates()
adjacency, node_coords, node_colors = build_connectome(selected, atlas_lookup)

# Positive r (Healthy > EarlyAD) is red; negative r is blue.
view = plotting.view_connectome(
    adjacency,
    node_coords,
    node_color=node_colors,
    node_size=NODE_SIZE,
    edge_threshold=0,
    edge_cmap=plt.cm.bwr,
)
display(view)

OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)
view.save_as_html(str(OUTPUT_HTML))
OUTPUT_EDGES_CSV.parent.mkdir(parents=True, exist_ok=True)
selected.drop(columns=["edge_key"]).to_csv(OUTPUT_EDGES_CSV, index=False)
print(
    f"{OUTPUT_HTML} | {len(selected)} undirected edges | "
    f"{len(node_coords)} cortical parcels"
)
